# 신부전 AI 진단 모델 (TabNet 기반)

## 개요
- **데이터**: Kaggle CKD Dataset (20,538개)
- **모델**: TabNet (Attention 기반 정형 데이터 딥러닝)
- **타겟**: 원본 eGFR → KDIGO 기준 5단계 분류
- **피처**: eGFR 포함 임상 수치 25개

## CKD 단계 기준 (KDIGO 2024)
| 단계 | eGFR 범위 | 설명 |
|------|-----------|------|
| Normal/Stage1 | ≥ 90 | 정상 또는 CKD 1단계 |
| Stage2 | 60~89 | 경미한 신기능 저하 |
| Stage3 | 30~59 | 중등도 신기능 저하 |
| Stage4 | 15~29 | 중증 신기능 저하 |
| Stage5 | < 15 | 신부전 (투석 필요) |


## 1. 라이브러리 임포트

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import os

def set_korean_font():
    candidates = [
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
    ]
    for fp in candidates:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
            plt.rcParams['font.family'] = fm.FontProperties(fname=fp).get_name()
            plt.rcParams['axes.unicode_minus'] = False
            return
    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()
warnings.filterwarnings('ignore')
print("라이브러리 임포트 완료")


## 2. 경로 및 상수 설정

In [ ]:
BASE_DIR     = Path('.')
DATA_PATH    = BASE_DIR / "datasets/kidney/kidney_disease_dataset.csv"
MODEL_DIR    = BASE_DIR / "models/kidney"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH   = MODEL_DIR / "tabnet_stage.pkl"
SCALER_PATH  = MODEL_DIR / "scaler_stage.pkl"
IMPUTER_PATH = MODEL_DIR / "imputer_stage.pkl"
ENCODER_PATH = MODEL_DIR / "encoder_stage.pkl"

KAGGLE_TO_STD = {
    'Age of the patient':                    'age',
    'Blood pressure (mm/Hg)':               'bp',
    'Specific gravity of urine':            'sg',
    'Albumin in urine':                     'al',
    'Sugar in urine':                       'su',
    'Random blood glucose level (mg/dl)':   'bgr',
    'Blood urea (mg/dl)':                   'bu',
    'Serum creatinine (mg/dl)':             'sc',
    'Sodium level (mEq/L)':                 'sod',
    'Potassium level (mEq/L)':              'pot',
    'Hemoglobin level (gms)':               'hemo',
    'Packed cell volume (%)':               'pcv',
    'White blood cell count (cells/cumm)':  'wc',
    'Red blood cell count (millions/cumm)': 'rc',
    'Red blood cells in urine':             'rbc',
    'Pus cells in urine':                   'pc',
    'Pus cell clumps in urine':             'pcc',
    'Bacteria in urine':                    'ba',
    'Hypertension (yes/no)':                'htn',
    'Diabetes mellitus (yes/no)':           'dm',
    'Coronary artery disease (yes/no)':     'cad',
    'Appetite (good/poor)':                 'appet',
    'Pedal edema (yes/no)':                 'pe',
    'Anemia (yes/no)':                      'ane',
    'Estimated Glomerular Filtration Rate (eGFR)': 'egfr',
}

NUMERIC_COLS = [
    'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc',
    'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'egfr',
]
CATEGORICAL_COLS = [
    'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm',
    'cad', 'appet', 'pe', 'ane',
]
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS
TARGET_COL   = 'ckd_stage'
STAGE_LABELS = ['Normal_Stage1', 'Stage2', 'Stage3', 'Stage4', 'Stage5']

print("상수 설정 완료")
print("수치형 피처:", len(NUMERIC_COLS), "개")
print("범주형 피처:", len(CATEGORICAL_COLS), "개")
print("전체 피처  :", len(FEATURE_COLS), "개")


## 3. 데이터 로드 및 탐색

In [ ]:
def assign_stage(egfr):
    if pd.isna(egfr):  return 'Normal_Stage1'
    if egfr >= 90:     return 'Normal_Stage1'
    elif egfr >= 60:   return 'Stage2'
    elif egfr >= 30:   return 'Stage3'
    elif egfr >= 15:   return 'Stage4'
    else:              return 'Stage5'

df = pd.read_csv(DATA_PATH)
df = df.rename(columns=KAGGLE_TO_STD)
df['egfr'] = pd.to_numeric(df['egfr'], errors='coerce')
df[TARGET_COL] = df['egfr'].apply(assign_stage)

print("데이터 크기:", df.shape)
print("\n단계별 분포:")
for stage in STAGE_LABELS:
    count = (df[TARGET_COL] == stage).sum()
    pct   = count / len(df) * 100
    print("  {:<15s}: {:5d}개 ({:.1f}%)".format(stage, count, pct))

df.head()


## 4. 데이터 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = [df[df[TARGET_COL]==s].shape[0] for s in STAGE_LABELS]
colors = ['#4CAF50','#2196F3','#FF9800','#F44336','#9C27B0']

axes[0].bar(STAGE_LABELS, counts, color=colors, alpha=0.8)
axes[0].set_title('CKD 단계별 데이터 분포', fontsize=13)
axes[0].set_xlabel('CKD 단계')
axes[0].set_ylabel('데이터 수')
for bar, count in zip(axes[0].patches, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 50,
                 str(count), ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

for stage, color in zip(STAGE_LABELS, colors):
    data = df[df[TARGET_COL]==stage]['egfr'].dropna()
    axes[1].hist(data, bins=30, alpha=0.6, color=color, label=stage)
axes[1].set_title('단계별 eGFR 분포', fontsize=13)
axes[1].set_xlabel('eGFR')
axes[1].set_ylabel('빈도')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 5. 데이터 전처리

In [ ]:
def preprocess(df, fit=True,
               imputer=None, scaler=None,
               cat_encoders=None, label_enc=None):
    df = df.copy()

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = (df[col].astype(str)
                               .str.strip()
                               .str.lower()
                               .replace({'?': np.nan, 'nan': np.nan})
                               .fillna('unknown'))

    if fit:
        cat_encoders = {}
        for col in CATEGORICAL_COLS:
            if col in df.columns:
                enc = LabelEncoder()
                df[col] = enc.fit_transform(df[col].astype(str))
                cat_encoders[col] = enc
    else:
        for col in CATEGORICAL_COLS:
            if col in df.columns and col in cat_encoders:
                enc = cat_encoders[col]
                df[col] = df[col].astype(str).map(
                    lambda x: enc.transform([x])[0]
                    if x in enc.classes_ else 0
                )

    feature_cols = [c for c in FEATURE_COLS if c in df.columns]
    X = df[feature_cols].values.astype(np.float32)

    if fit:
        imputer = SimpleImputer(strategy='median')
        X = imputer.fit_transform(X)
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    else:
        X = imputer.transform(X)
        X = scaler.transform(X)

    y = None
    if TARGET_COL in df.columns:
        if fit:
            label_enc = LabelEncoder()
            label_enc.fit(STAGE_LABELS)
            y = label_enc.transform(df[TARGET_COL])
        else:
            y = label_enc.transform(df[TARGET_COL])

    return X, y, imputer, scaler, cat_encoders, label_enc, feature_cols

X, y, imputer, scaler, cat_encoders, label_enc, feature_cols = preprocess(df)
print("전처리 완료")
print("X shape :", X.shape)
print("클래스  :", label_enc.classes_)


## 6. SMOTE 오버샘플링

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("SMOTE 적용 전:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print("  {:<15s}: {}개".format(label_enc.classes_[u], c))

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("\nSMOTE 적용 후:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print("  {:<15s}: {}개".format(label_enc.classes_[u], c))

print("\n학습 세트:", X_train.shape)
print("검증 세트:", X_val.shape)


## 7. TabNet 모델 학습

In [ ]:
model = TabNetClassifier(
    n_d=64, n_a=64,
    n_steps=5,
    gamma=1.5,
    n_independent=3,
    n_shared=3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=1e-3, weight_decay=1e-5),
    scheduler_params=dict(step_size=30, gamma=0.9),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',
    verbose=10,
    device_name='cpu',
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['accuracy'],
    max_epochs=300,
    patience=30,
    batch_size=512,
    virtual_batch_size=256,
    weights=1,
)

print("TabNet 학습 완료")


## 8. 모델 평가

In [ ]:
y_pred     = model.predict(X_val)
pred_proba = model.predict_proba(X_val)
acc        = accuracy_score(y_val, y_pred)

print("검증 정확도: {:.4f} ({:.2f}%)".format(acc, acc*100))
print()
print(classification_report(y_val, y_pred, target_names=label_enc.classes_))


## 9. 혼동행렬 시각화

In [ ]:
kor_labels = ['정상/1단계','2단계','3단계','4단계','5단계']
cm      = confusion_matrix(y_val, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=kor_labels, yticklabels=kor_labels, ax=axes[0])
axes[0].set_title('혼동행렬 (실제 개수)', fontsize=13)
axes[0].set_ylabel('실제 클래스')
axes[0].set_xlabel('예측 클래스')

sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens',
            xticklabels=kor_labels, yticklabels=kor_labels,
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('혼동행렬 (정규화)', fontsize=13)
axes[1].set_ylabel('실제 클래스')
axes[1].set_xlabel('예측 클래스')

fig.suptitle('CKD 단계 분류 혼동행렬', fontsize=15)
plt.tight_layout()
plt.show()


## 10. 피처 중요도 (TabNet Attention)

In [ ]:
FEAT_KOR = {
    'age':'나이','bp':'혈압','sg':'소변비중','al':'알부민뇨',
    'su':'당뇨','bgr':'혈당','bu':'혈중요소','sc':'혈청크레아티닌',
    'sod':'나트륨','pot':'칼륨','hemo':'헤모글로빈','pcv':'적혈구용적',
    'wc':'백혈구수','rc':'적혈구수','egfr':'eGFR',
    'rbc':'적혈구(소변)','pc':'고름세포','pcc':'고름세포군집',
    'ba':'박테리아','htn':'고혈압','dm':'당뇨병','cad':'관상동맥',
    'appet':'식욕','pe':'부종','ane':'빈혈',
}

importance = model.feature_importances_
feat_imp   = pd.Series(importance, index=feature_cols).sort_values()
kor_index  = [FEAT_KOR.get(f, f) for f in feat_imp.index]

colors_bar = ['#d32f2f' if v >= feat_imp.quantile(0.75)
              else '#1976d2' if v >= feat_imp.quantile(0.5)
              else '#757575'
              for v in feat_imp.values]

fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(range(len(feat_imp)), feat_imp.values, color=colors_bar)
ax.set_yticks(range(len(feat_imp)))
ax.set_yticklabels(kor_index, fontsize=10)
ax.set_xlabel('중요도 (TabNet Attention)', fontsize=11)
ax.set_title('피처 중요도', fontsize=13)
for bar, val in zip(bars, feat_imp.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            '{:.4f}'.format(val), va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("1위:", FEAT_KOR.get(feat_imp.index[-1]), "{:.4f}".format(feat_imp.values[-1]))
print("2위:", FEAT_KOR.get(feat_imp.index[-2]), "{:.4f}".format(feat_imp.values[-2]))
print("3위:", FEAT_KOR.get(feat_imp.index[-3]), "{:.4f}".format(feat_imp.values[-3]))


## 11. 모델 저장

In [ ]:
with open(MODEL_PATH,  'wb') as f: pickle.dump(model, f)
with open(SCALER_PATH, 'wb') as f: pickle.dump(scaler, f)
with open(IMPUTER_PATH,'wb') as f: pickle.dump(imputer, f)
with open(ENCODER_PATH,'wb') as f:
    pickle.dump({
        'cat':      cat_encoders,
        'label':    label_enc,
        'features': feature_cols,
    }, f)

print("모델 저장 완료")
print(str(MODEL_PATH))


## 12. 추론 테스트

In [ ]:
STAGE_INFO = {
    'Normal_Stage1': {'desc': '정상/CKD 1단계 (eGFR >= 90)',     'severity': 'normal'},
    'Stage2':        {'desc': 'CKD 2단계 (eGFR 60~89)',           'severity': 'mild'},
    'Stage3':        {'desc': 'CKD 3단계 (eGFR 30~59)',           'severity': 'moderate'},
    'Stage4':        {'desc': 'CKD 4단계 (eGFR 15~29)',           'severity': 'severe'},
    'Stage5':        {'desc': 'CKD 5단계 - 투석 필요 (eGFR < 15)','severity': 'critical'},
}

test_cases = [
    {
        "name": "CKD 4단계 환자",
        "data": {
            'age':48,'bp':80,'sg':1.01,'al':3,'su':0,
            'bgr':121,'bu':36,'sc':7.2,'sod':130,'pot':4.5,
            'hemo':9.5,'pcv':29,'wc':7800,'rc':3.2,'egfr':18.0,
            'rbc':'abnormal','pc':'abnormal','pcc':'notpresent',
            'ba':'notpresent','htn':'yes','dm':'yes','cad':'no',
            'appet':'poor','pe':'yes','ane':'yes'
        }
    },
    {
        "name": "정상 환자",
        "data": {
            'age':30,'bp':70,'sg':1.02,'al':0,'su':0,
            'bgr':90,'bu':18,'sc':0.8,'sod':138,'pot':4.0,
            'hemo':15.5,'pcv':46,'wc':7200,'rc':5.2,'egfr':95.0,
            'rbc':'normal','pc':'normal','pcc':'notpresent',
            'ba':'notpresent','htn':'no','dm':'no','cad':'no',
            'appet':'good','pe':'no','ane':'no'
        }
    },
]

for case in test_cases:
    df_t = pd.DataFrame([case["data"]])
    for col in NUMERIC_COLS:
        if col in df_t.columns:
            df_t[col] = pd.to_numeric(df_t[col], errors='coerce')
    for col in CATEGORICAL_COLS:
        if col in df_t.columns and col in cat_encoders:
            df_t[col] = df_t[col].astype(str).str.strip().str.lower()
            enc = cat_encoders[col]
            df_t[col] = df_t[col].map(
                lambda x: enc.transform([x])[0] if x in enc.classes_ else 0)
    for col in feature_cols:
        if col not in df_t.columns:
            df_t[col] = np.nan

    X_t = df_t[feature_cols].values.astype(np.float32)
    X_t = imputer.transform(X_t)
    X_t = scaler.transform(X_t)

    pred_idx   = model.predict(X_t)[0]
    pred_proba = model.predict_proba(X_t)[0]
    prediction = label_enc.classes_[pred_idx]
    confidence = pred_proba[pred_idx]
    info       = STAGE_INFO[prediction]

    print("=" * 50)
    print("환자    :", case["name"])
    print("예측    :", prediction)
    print("설명    :", info["desc"])
    print("신뢰도  : {:.4f}".format(confidence))
    print("심각도  :", info["severity"])
    print("단계별 확률:")
    for cls, prob in zip(label_enc.classes_, pred_proba):
        bar = chr(9608) * int(prob * 20)
        print("  {:<15s}: {:.4f} {}".format(cls, prob, bar))
    print()


## 13. 최종 요약

In [ ]:
print("=" * 60)
print("신부전 TabNet 모델 최종 요약")
print("=" * 60)
print("데이터셋   : Kaggle CKD Dataset (20,538개)")
print("모델       : TabNet (Attention 기반)")
print("피처 수    :", len(feature_cols), "(eGFR 포함)")
print("타겟       : KDIGO 5단계 분류")
print("검증 정확도: {:.4f} ({:.2f}%)".format(acc, acc*100))
print()
print("[피처 중요도 Top 3]")
print("1위:", FEAT_KOR.get(feat_imp.index[-1]), "{:.4f}".format(feat_imp.values[-1]))
print("2위:", FEAT_KOR.get(feat_imp.index[-2]), "{:.4f}".format(feat_imp.values[-2]))
print("3위:", FEAT_KOR.get(feat_imp.index[-3]), "{:.4f}".format(feat_imp.values[-3]))
